In [12]:
from cgra import *
from kernels import *
import random

In [13]:
kernel_name = "benchmarks/disco-vs-oe/transpose-scale"
version = ""

In [14]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [15]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [16]:
# Data
def configMemory(A_data, rowsA, colsA, scale_f):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # ------------------------------------------
    # CONFIGURATION
    # ------------------------------------------
    # nItLoopBlocksCols     A[0][1]               A[0][2]     A[0][3]
    # A[1][0]               nItLoopBlocksRows     A[1][2]     A[1][3]
    # A[2][0]               A[2][1]               A[2][2]     A[2][3]
    # A[3][0]               A[3][1]               A[3][2]     A[3][3]
    # ------------------------------------------
    # B[0][0]               B[0][1]               B[0][2]     B[0][3]
    # B[1][0]               B[1][1]               B[1][2]     B[1][3]
    # B[2][0]               B[2][1]               B[2][2]     B[2][3]
    # B[3][0]               B[3][1]               B[3][2]     B[3][3]
    # -------------------------------------------
    # scale_f               scale_f               scale_f     scale_f
    # scale_f               scale_f               scale_f     scale_f
    # scale_f               scale_f               scale_f     scale_f
    # scale_f               scale_f               scale_f     scale_f
    # -------------------------------------------
    # -                     colsIn                colsOut     16*(nBlocksColsi + 1)
    # 16*(nBlocksColsi + 1) -                     colsIn      colsOut
    # colsOut               16*(nBlocksColsi + 1) -           colsIn
    # colsIn                colsOut               16*(nBlocksColsi + 1) -
    # -------------------------------------------

    colsB = rowsA # B = At
    nItLoopBlocksCols = colsA // CGRA_N_COLS
    nItLoopBlocksRows = rowsA // CGRA_N_ROWS

    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA * colsA * 4

    config_vals_col0 = [
        nItLoopBlocksCols,
        first_addr_A + (1*colsA + 0)*4,
        first_addr_A + (2*colsA + 0)*4,
        first_addr_A + (3*colsA + 0)*4,

        first_addr_B + (0*colsB + 0)*4,
        first_addr_B + (0*colsB + 1)*4,
        first_addr_B + (0*colsB + 2)*4,
        first_addr_B + (0*colsB + 3)*4,

        scale_f, scale_f, scale_f, scale_f,

        16*(nItLoopBlocksCols + 1),
        colsB,
        colsA
    ]



    config_vals_col1 = [
        first_addr_A + (0*colsA + 1)*4,
        nItLoopBlocksRows,
        first_addr_A + (2*colsA + 1)*4,
        first_addr_A + (3*colsA + 1)*4,

        first_addr_B + (1*colsB + 0)*4,
        first_addr_B + (1*colsB + 1)*4,
        first_addr_B + (1*colsB + 2)*4,
        first_addr_B + (1*colsB + 3)*4,

        scale_f, scale_f, scale_f, scale_f,

        colsA,
        16*(nItLoopBlocksCols + 1),
        colsB
    ]



    config_vals_col2 = [
        first_addr_A + (0*colsA + 2)*4,
        first_addr_A + (1*colsA + 2)*4,
        first_addr_A + (2*colsA + 2)*4,
        first_addr_A + (3*colsA + 2)*4,

        first_addr_B + (2*colsB + 0)*4,
        first_addr_B + (2*colsB + 1)*4,
        first_addr_B + (2*colsB + 2)*4,
        first_addr_B + (2*colsB + 3)*4,

        scale_f, scale_f, scale_f, scale_f,

        colsB,
        colsA,
        16*(nItLoopBlocksCols + 1)
    ]


    config_vals_col3 = [
        first_addr_A + (0*colsA + 3)*4,
        first_addr_A + (1*colsA + 3)*4,
        first_addr_A + (2*colsA + 3)*4,
        first_addr_A + (3*colsA + 3)*4,

        first_addr_B + (3*colsB + 0)*4,
        first_addr_B + (3*colsB + 1)*4,
        first_addr_B + (3*colsB + 2)*4,
        first_addr_B + (3*colsB + 3)*4,

        scale_f, scale_f, scale_f, scale_f,

        16*(nItLoopBlocksCols + 1),
        colsB,
        colsA
    ]

    addr = 0
    for cfg in [config_vals_col0, config_vals_col1,
                config_vals_col2, config_vals_col3]:
        kernel_add_memory_region(kernel_name, addr, cfg, version=version)
        addr += len(cfg) * 4

    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)

    return [
        0,
        len(config_vals_col0) * 4,
        (len(config_vals_col0) + len(config_vals_col1)) * 4,
        (len(config_vals_col0) + len(config_vals_col1) + len(config_vals_col2)) * 4
    ]

In [17]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT", "R1", "INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [18]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [19]:
def transpose_and_scale_cpu(A_data, rowsA, colsA, scale_f):
    out = [0 for _ in range(rowsA * colsA)]
    for i in range(rowsA):
        for j in range(colsA):
            out[j*rowsA + i] = A_data[i*colsA + j] >> scale_f
    return out

In [20]:
# Test dimensions (4xXx4)
# 4x4 OK
# 4x8 OK
# 8x4 Err
rowsA = 8
colsA = 4
scale_f = 0

A_data = [random.randint(-10, 10) for _ in range(rowsA * colsA)]

A_data_cpy = A_data.copy()

#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, rowsA, colsA, scale_f)


expected_res = transpose_and_scale_cpu(A_data, rowsA, colsA, scale_f)
printAsMatrix(expected_res, colsA, rowsA)

[8, 9, 8, 1, 6, 9, 1, -1]
[-9, 9, -4, 9, -3, -2, -9, 2]
[-6, -8, -6, -4, -1, 6, 0, 1]
[-9, -8, -3, -2, 1, -7, 0, -6]


In [21]:
runKernel(load_addrs, max_it=20000)

Instr =  0 ( 0 )
[   1, 20004, 20008, 20012]    [   0,    0,    0,    0]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20016,    2, 20024, 20028]    [   0,    0,    0,    0]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20032, 20036, 20040, 20044]    [   0,    0,    0,    0]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
[20048, 20052, 20056, 20060]    [   0,    0,    0,    0]    [LWD R0  4, LWD R0  4, LWD R0  4, LWD R0  4]    
Aprox cycles this pc: 17
-------
Instr =  1 ( 1 )
[20128, 20160, 20192, 20224]    [20128, 20160, 20192, 20224]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[20132, 20164, 20196, 20228]    [20132, 20164, 20196, 20228]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[20136, 20168, 20200, 20232]    [20136, 20168, 20200, 20232]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
[20140, 20172, 20204, 20236]    [20140, 20172, 20204, 20236]    [LWD R1  4, LWD R1  4, LWD R1  4, LWD R1  4]    
Aprox cycles this pc: 17
-------
Instr =  2 ( 2

In [22]:
# Get result from CGRA
first_addr_out = first_addr + rowsA * colsA * 4
result = getResult(first_addr_out, first_addr_out + rowsA * colsA * 4, rowsA, colsA)
# Process estra rows/cols


# Get cpu output
expected_res = transpose_and_scale_cpu(A_data, rowsA, colsA, scale_f)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors) + " out of " + str(rowsA*colsA))
    print("A: ")
    printAsMatrix(A_data_cpy, rowsA, colsA)
    print("Expected: ")
    printAsMatrix(expected_res, colsA, rowsA)
    print("CGRA: ")
    printAsMatrix(result, colsA, rowsA)
else:
    print("OK")



Err: 14 out of 32
A: 
[8, -9, -6, -9]
[9, 9, -8, -8]
[8, -4, -6, -3]
[1, 9, -4, -2]
[6, -3, -1, 1]
[9, -2, 6, -7]
[1, -9, 0, 0]
[-1, 2, 1, -6]
Expected: 
[8, 9, 8, 1, 6, 9, 1, -1]
[-9, 9, -4, 9, -3, -2, -9, 2]
[-6, -8, -6, -4, -1, 6, 0, 1]
[-9, -8, -3, -2, 1, -7, 0, -6]
CGRA: 
[8, 9, 8, 1, 0, 0, 0, 0]
[-9, 9, -4, 9, 0, 0, 0, 0]
[-6, -8, -6, -4, 0, 0, 0, 0]
[-9, -8, -3, -2, 0, 0, 0, 0]
